# llm-inference — vLLM demo on a free GPU

This notebook serves **Qwen/Qwen3-0.6B** with the real [vLLM](https://github.com/vllm-project/vllm)
engine on Colab's free T4 GPU and exercises it through the OpenAI-compatible API: a first request,
**continuous batching**, **prefix caching**, and the live Prometheus `/metrics`.

> **Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Repo: https://github.com/giannisp09/llm-inference


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install vLLM

A few minutes the first time. vLLM pulls in CUDA wheels; that's why this needs Linux + GPU
(and why you can't `pip install vllm` on a Mac).

In [ ]:
!pip install -q vllm openai

## 3. Start the vLLM server in the background

This is exactly what `./scripts/serve.sh qwen3-0.6b` runs in the repo — `vllm serve` with the
flags from `configs/qwen3-0.6b.yaml`. We launch it as a subprocess and tail the log.

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-0.6B",
     "--dtype", "bfloat16", "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.90", "--enable-prefix-caching",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT,
)

BASE_URL = "http://localhost:8000"
print("Waiting for vLLM to load the model (downloads weights on first run)...")
for i in range(120):
    try:
        if requests.get(f"{BASE_URL}/v1/models", timeout=5).status_code == 200:
            MODEL = requests.get(f"{BASE_URL}/v1/models").json()["data"][0]["id"]
            print("Ready — model:", MODEL); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print("Did not come up — last log lines:")
    print(open("vllm.log").read()[-2000:])

## 4. Your first request (OpenAI client)

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="unused")

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is PagedAttention in one sentence?"}],
    max_tokens=80, temperature=0.7,
    extra_body={"top_k": 20, "chat_template_kwargs": {"enable_thinking": False}},
)
print(resp.choices[0].message.content)
print("\nusage:", resp.usage)

## 5. Continuous batching

Fire 5 requests at once. vLLM schedules at the token level, so they all generate together
instead of queueing one-by-one.

In [ ]:
import concurrent.futures, time

prompts = ["What is quantization?", "Explain KV caching briefly.",
           "What is continuous batching?", "Why is LLM inference memory-bound?",
           "What is PagedAttention?"]

def ask(p):
    return client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": p}],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})

t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
    for f in concurrent.futures.as_completed([pool.submit(ask, p) for p in prompts]):
        r = f.result(); print(f"  {r.usage.completion_tokens:3d} tokens")
print(f"\nAll 5 done in {time.time()-t0:.2f}s")

## 6. Prefix caching

Five requests sharing one system prompt. The first pays full prefill; the rest reuse the cached
KV blocks for the prefix. Watch `prefix_cache_hits_total` climb.

In [ ]:
def metrics():
    m = {}
    for line in requests.get(f"{BASE_URL}/metrics").text.splitlines():
        if line and not line.startswith("#"):
            try: m[line.split("{")[0].split()[0]] = float(line.split()[-1])
            except (ValueError, IndexError): pass
    return m

SYSTEM = ("You are a concise teaching assistant for a course on LLM optimization. "
          "Answer in one or two sentences.")
questions = ["What is weight quantization?", "How does vLLM handle memory?",
             "What is continuous batching?", "Why use prefix caching?", "What is GPTQ?"]

before = metrics()
for q in questions:
    client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
        max_tokens=60, extra_body={"chat_template_kwargs": {"enable_thinking": False}})
after = metrics()
print("prefix_cache_queries:", after.get("vllm:prefix_cache_queries_total"))
print("prefix_cache_hits:   ", after.get("vllm:prefix_cache_hits_total"))

## 7. Live serving metrics

In [ ]:
m = metrics()
for k in ["vllm:num_requests_running", "vllm:gpu_cache_usage_perc",
          "vllm:prompt_tokens_total", "vllm:generation_tokens_total",
          "vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total"]:
    if k in m: print(f"  {k.replace('vllm:',''):<30} {m[k]:g}")

## Done

You served a real LLM with vLLM and exercised continuous batching, prefix caching, and metrics —
the same flow the repo runs in Docker/Kubernetes, just on a free GPU.

Next: clone the repo on a bigger GPU and serve `llama3-8b`, or bring up the full
Prometheus + Grafana stack with `docker compose up`.

In [ ]:
server.terminate()  # stop the vLLM server